# A1 Assignment — Train 50k Vocabulary RNN on Colab

This notebook trains the **50,000-word** RNN language model using the same
hyperparameters as the successful 10k experiment.

**Before running:**
1. Runtime → **Change runtime type** → **T4 GPU** (or better)
2. Upload your project files (see Step 2)

**After training:** download `trainer_output_50k.zip` and extract it into
your local `a1/` folder, then run `python compare_experiments.py` locally.

## Step 1 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: No GPU detected. Training will be very slow.")
    print("Go to Runtime → Change runtime type → GPU, then re-run this cell.")

## Step 2 — Install dependencies

In [ ]:
!pip install -q nltk transformers datasets accelerate

## Step 3 — Get project files

Pick **one** of the options below.

### Option A (recommended): Upload a zip from your laptop

On your Mac, from the `a1/` directory run:

```bash
cd ~/dl4nlp-assignments/a1
zip -r a1_colab_upload.zip \
  A1_skeleton.py part4.py evaluate_model.py \
  train.txt val.txt
```

Then upload `a1_colab_upload.zip` in the next cell.

In [ ]:
from google.colab import files
import zipfile, os

WORK_DIR = "/content/a1"
os.makedirs(WORK_DIR, exist_ok=True)

print("Upload a1_colab_upload.zip (or cancel and use Option B below)")
uploaded = files.upload()  # pick a1_colab_upload.zip

zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(WORK_DIR)

print("Extracted to", WORK_DIR)
print(os.listdir(WORK_DIR))

### Option B: Clone from GitHub + upload data files

Use this if your code is on GitHub but `train.txt` / `val.txt` are too large for the repo.

In [ ]:
# Uncomment and edit if using Option B instead of Option A.

# !git clone https://github.com/641bill/dl4nlp-assignments.git /content/dl4nlp-assignments
# WORK_DIR = "/content/dl4nlp-assignments/a1"
# %cd {WORK_DIR}

# from google.colab import files
# print("Upload train.txt and val.txt")
# uploaded = files.upload()
# print("Files in working dir:", os.listdir(WORK_DIR))

## Step 4 — Verify files

In [ ]:
%cd /content/a1

required = [
    "A1_skeleton.py",
    "part4.py",
    "evaluate_model.py",
    "train.txt",
    "val.txt",
]

missing = [f for f in required if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

for f in required:
  size_mb = os.path.getsize(f) / 1e6
  print(f"  {f:25s}  {size_mb:8.1f} MB")

print("\nAll required files present.")

## Step 5 — (Optional) Quick sanity check

Runs 1 epoch on 128 training examples. Skip if you already verified locally.

In [ ]:
# Set RUN_SANITY_CHECK = True in part4.py, run once, then set back to False.
# Or just skip this cell and go straight to full training.

import re
with open("part4.py") as f:
    src = f.read()
src_sanity = re.sub(
    r"RUN_SANITY_CHECK = False",
    "RUN_SANITY_CHECK = True",
    src,
    count=1,
)
with open("part4_sanity.py", "w") as f:
    f.write(src_sanity)

!python part4_sanity.py

## Step 6 — Full training (50k vocab, 3 epochs)

Expected runtime on T4 GPU: **~30–60 minutes** (much faster than local MPS).

Configuration (from `part4.py`):
- `MAX_VOC_SIZE = 50_000`
- `OUTPUT_DIR = trainer_output_50k`
- embedding=128, hidden=256, lr=1e-3, batch=32, epochs=3

In [ ]:
%cd /content/a1

# Ensure sanity check is off for the real run
import re
with open("part4.py") as f:
    src = f.read()
if "RUN_SANITY_CHECK = True" in src:
    src = src.replace("RUN_SANITY_CHECK = True", "RUN_SANITY_CHECK = False", 1)
    with open("part4.py", "w") as f:
        f.write(src)

!python part4.py

## Step 7 — Evaluate on validation set

In [ ]:
%cd /content/a1
!python evaluate_model.py trainer_output_50k

## Step 8 — Download results

Download `trainer_output_50k.zip` to your laptop, then:

```bash
cd ~/dl4nlp-assignments/a1
unzip ~/Downloads/trainer_output_50k.zip
python compare_experiments.py
```

In [ ]:
import shutil
from google.colab import files

OUTPUT_DIR = "trainer_output_50k"
assert os.path.isdir(OUTPUT_DIR), f"{OUTPUT_DIR} not found — training may have failed"
assert os.path.exists(f"{OUTPUT_DIR}/model.safetensors"), "model weights missing"
assert os.path.exists(f"{OUTPUT_DIR}/tokenizer.pkl"), "tokenizer missing"

shutil.make_archive("trainer_output_50k", "zip", OUTPUT_DIR)
print("Created trainer_output_50k.zip")
files.download("trainer_output_50k.zip")

## (Optional) Save to Google Drive

Use this if the download is unreliable or you want a backup.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")

dest = "/content/drive/MyDrive/dl4nlp/trainer_output_50k"
if os.path.exists(dest):
    shutil.rmtree(dest)
shutil.copytree("trainer_output_50k", dest)
print(f"Copied to {dest}")